# Lesson 1, Part 4

Let's teach Python to calculate the derivative of *y=x*(7-*x*), 
without telling it that the true answer is 7-2*x*. 
We will start with *dx*=1, keep dividing *dx* by 10, 
and look for the limit of *dy/dx*.

In [ ]:
perim,x,dx = 14,3,1
y = x * (perim/2 - x)
for iter in range(7):
    dx /= 10
    x1 = x + dx
    y1 = x1 * (perim/2 - x1)
    dydx = (y1 - y) / dx
    print(round(dx,15), round(dydx,10))

When the nudges *dx* get really tiny, the estimates stabilize and then deteriorate. 
The problem is the rounding error from computing *dy* as *y*+*dy* minus *y*.
We should never rely on float computations to specify a limit exactly.
We need algebra to substitute *y - y* = 0 and *dx/dx* = 1 before we take limits.

In [ ]:
perim,x,dx = 14,3,1E-7
y = x * (perim/2 - x)
for iter in range(8):
    dx /= 10
    x1 = x + dx
    y1 = x1 * (perim/2 - x1)
    dydx = (y1 - y) / dx
    print(round(dx,15), round(dydx,10))

Computers are best used to refine approximations, of which the most famous is the ratio $\pi$ of a circle's perimeter *P* to its diameter *D*. 
The largest polygon that can fit inside the circle has distance *D* between opposite vertices,
while the smallest polygon that can enclose the circle has distance *D* between opposite sides.
$\pi$ must lie between the two *P/D* ratios. To illustrate this, we can use the turtle command **circle**.

In [ ]:
%pip install ipycc
from ipycc.turtle import Turtle, showscreen, clearscreen
showscreen()
clearscreen()
t = Turtle()
radius = 50
# draw a circle of radius 50, with first point at bottom of circle
t.circle(radius)   # actually, it's a regular polygon with barely visible vertices
# improve the approximation by increasing number of sides, called "steps" by turtle
t.color('orange')
radius = 75  # let's enlarge the circle to facilitate comparisons
t.circle(radius,steps=100)  # regular polygon with 100 sides better resembles circle but takes longer to draw
# turtle circles with few sides are clearly polygons
t.color('green')
t.circle(75,steps=4)  # square with diagonal 150
# we can use this to draw regular polygons with maximum distance of radius from center
t.color('red')
t.circle(100,steps=8)  # regular octagon with maximum diagonal 200

By nesting a hexagon inside a circle inside a hexagon, 
we can bound  𝜋  between 3 and 3.5.

In [ ]:
showscreen()
t.reset()
radius = 100
t.pencolor('blue')
t.fillcolor('cyan')
t.begin_fill()
side = 1.155 * radius   # the precise multiplier is 2/sqrt(3) 
t.backward(side/2)        # will put middle of first side at home
for iter in range(6):
    t.forward(side)
    t.left(60)
t.home()
t.end_fill()
t.fillcolor('yellow')
t.begin_fill()
t.circle(radius)
t.end_fill()
t.fillcolor('cyan')
t.begin_fill()
t.circle(100, steps=6)
t.end_fill()

By repeatedly doubling sides, Archimedes proved that  𝜋  lies
between 3 10/71 ≈3.1408  and  3 1/7≈3.1429.
Let's use turtle to estimate even tighter bounds.
For an even-sided polygon with horizontal bottom,
the height at the horizontal top tells us  𝐷  for the largest nested circle. 
The ratio of perimeter  𝑃  to  𝐷  sets an upper bound for  𝜋 .
For 96 sides, our answer is similar to Archimedes'.

In [ ]:
showscreen()
t.reset()
t.speed('fastest')
# upper bound on π using 96 sides
perim,n = 600,96
for k in range(n):
    if k == n/2: # while k is integer and n/2==48.0 is float, Python understands that 48==48.0
        height = t.ycor()
    t.forward(perim / n)
    t.left(360 / n)
pi_high = perim / height
print(pi_high)

If we rotate the polygon by half a turn, the polygon juts out at bottom and top. 
Its height now tells us the diameter of the smallest circle that can 
nest the polygon. The ratio of perimeter to height gives a lower bound for $\pi$ .

In [ ]:
showscreen()
t.reset()
t.speed('fastest')
# lower bound on π using 96 sides
perim,n = 600,96
t.left(180 / n)
for k in range(n):
    if k == n/2: # while k is integer and n/2==48.0 is float, Python understands that 48==48.0
        height = t.ycor()
    t.forward(perim / n)
    t.left(360 / n)
pi_low = perim / height
print(pi_low)

To avoid redrawing the polygon with a half turn, just calculate the distance of the farthest vertex from the origin. 
Also, once we get to that vertex, which has index  𝑛/2 , we don’t need to draw the rest of the polygon.
This lets us compress the previous codes.

In [ ]:
showscreen()
t.reset()
# lower bound on π using 96 sides
perim,n = 600,96
for k in range(n//2):  # n//2, not n/2, as range() needs an integer
    t.forward(perim / n)
    t.left(360 / n)
pi_high = perim / t.ycor()
pi_low = perim / t.distance(0,0)
print(pi_low,'< π <',pi_high)

Our bounds are a bit tighter than Archimedes’ bounds due to the bounds he chose for square roots. 
To tighten them further, let’s keep doubling the number of sides. The results for 1536 sides specify $\pi$ to five decimal digits,
with an error of less than two parts in a million. Granted, we haven't proved that turtle computed them right.
Still, it’s fun to watch convergence in action, especially when a computer does all the hard work.

In [ ]:
showscreen()
t.reset()
t.speed('fastest')    # but this will still be very slow for n>1000
perim,n = 600,48
pi_bounds = []
for iter in range(5):
    n *= 2  # double n every iteration
    t.home()
    for k in range(n//2):
        t.forward(perim / n)
        t.left(360 / n)
    pi_high = perim / t.ycor()
    pi_low = perim / t.distance(0,0)
    pi_bounds.append([n, pi_low, pi_high])
    print(pi_bounds[iter])

Next let's examine Python methods for working with sequences. Here are three different ways of creating a list of lists. 
The first two we've seen before. The third wraps everything into a single command known as a **list comprehension**. 
It creates an empty list that the loop inside fills, and the loop is rewritten to put the command before the header.

In [ ]:
# append rows of lists to an empty list
sample = []
for k in range(5):
    row = [4*k,4*k+1,4*k+2,4*k+3]
    print('row',k,'is',row)
    sample.append(row) 

# replace each element of a placeholder list with a list
sample1 = list(range(5))
for k in range(5):
    row = list(range(4*k,4*k+4)) 
    sample1[k] = row
    
# do this in one line with a list comprehension
sample2 = [[4*k,4*k+1,4*k+2,4*k+3] for k in range(5)]

print('Do the 3 methods generate the same results?', sample2 == sample1 == sample)
print('sample printed without each row on new line')
print(sample)

Now let's explore some lookup methods for lists that work with tuples too. 
Suppose we want to retrieve a row near the end but can't recall how many rows there are. 
While we can use **len**, we don't need to. 
Python indexes items from the end back to the beginning 
using negative integers, starting with -1.

In [ ]:
sample = [[4*k,4*k+1,4*k+2,4*k+3] for k in range(5)]
n = len(sample) - 1
print('row ',n,':',sample[n])  # last index is always one less than length
print('row -1:',sample[-1])   # simpler and gives the same result
n = len(sample) - 2
print('Does row -2 match row',n,'?',sample[-2] == sample[n])
print('Element at index [-2][-1] is',sample[-2][-1])    # works in multiple dimensions

To access regular blocks of data, use **slice(start,stop,step)**. 
Like range(),  the sequence runs start,start+step,start+2 step,... and halts just before stop. 
But the preferred way to write slice() is **start:stop:step**. 
The defaults used for omitted values are 0 for start, **len** for stop, and 1 for step.

In [ ]:
sample = [[4*k,4*k+1,4*k+2,4*k+3] for k in range(5)]
print(sample[slice(0,2,1)])  # print first two rows of sample
print(sample[0:2:1])         # simpler expession using colons :
print(sample[:2])            # even simpler expression
print(sample[2::3])          # print rows with indices 2 and 5 
print(sample[-3::2])         # print rows with indices -3 and -1

Sometimes we want to slice out a block of selected rows and columns, 
say [[8,9],[13,14]]. 
Unfortunately, sample[1:3][3:] won't work, as the [3:] 
looks for the fourth row of sample[1:3] rather than the fourth column. 
Instead we need to use a **for** loop or **list comprehension**.

In [ ]:
sample = [[4*k,4*k+1,4*k+2,4*k+3] for k in range(5)]
print(sample[1:3][3:])    # looks for the fourth row of the two-row sample0[1:3] and doesn't find one

# for loop can extract columns [3:] of each row
block1 = list(range(1,3))
for k in range(len(block1)):
    block1[k] = sample[k+1][3:]

# list comprehension extracts columns in a single line
block2 = [row[3:] for row in sample[1:3]]

print(block2,'matches',block1)

We can delete any element of a list or replace it with multiple elements, 
and the elements don't even need to be the same type.

In [ ]:
sample = [[4*k,4*k+1,4*k+2,4*k+3] for k in range(5)]
del sample[2][0]    # deletes the element at index [2][0]
del sample[3][2:4]  # deletes elements at index [3][2] and [3][3]
sample[4] = ['dog','cat']  # replaces row[4] with ['dog','cat']
print(sample)

A Python **dictionary** offers another way to store a labeled list or list of lists.
We won't use it much in this course but it's good to know.  

The first line below defines a dictionary 'num_sides' with two **keys** 
'triangle' and 'quadrilateral' and corresponding **values** of 3 and 4. 
The colons : link key:value pairs. The curly brackets { } are essential, so as not to confuse with a tuple or a list.

In [ ]:
num_sides = {'triangle':3, 'quadrilateral':4}                # defines dictionary with two key:value pairs
print('A triangle has',num_sides['triangle'],'sides')
num_sides1 = dict([('triangle',3),('quadrilateral',4)])      # can also form dictionary as list of tuples
print('A quadrilateral has',num_sides1['quadrilateral'],'sides')
num_sides['pentagon'] = 5                                    # easiest way to add single key:value pair 
num_sides.update({'hexagon':6,'octagon':8})                  # add multiple key:value pairs
print("The key 'hexagon' has value",num_sides['hexagon'])    # need separate "" and '' apostrophes
# print('The key 'hexagon' has value',num_sides['hexagon'])  # this will generate error message

Dictionaries have methods to list keys, values, and key:value pairs.

In [ ]:
num_sides = {'triangle':3, 'quadrilateral':4,'pentagon':5,'hexagon':6,'octagon':8} 
list_keys = list(num_sides.keys())     # form list of all the keys
print('The keys are',list_keys)
list_values = list(num_sides.values()) # form list of all the values
print('The values are',list_values)
list_items = list(num_sides.items())   # form list of tuples of all key:value pairs
print(list_items)

Keys don't have to be strings and values don't have to be numeric. 
Either can be tuples. 
In fact, there are few restrictions other than that keys can't be lists, 
so as not to inadvertently change them. 
The example below switches keys with values and adds abbreviations.

In [ ]:
num_sides = {'triangle':3, 'quadrilateral':4,'pentagon':5,'hexagon':6,'octagon':8} 
name_polygon = {}  # define an empty dictionary
for key,value in num_sides.items():
    abbreviation = key[:3]   # strings can be indexed like lists or tuples
    name_polygon[value] = [key,abbreviation]  
print(name_polygon)
print(name_polygon[6])      # value will have two elements
print(name_polygon[6][1])   # give only the abbreviation

Here we store high and low bounds for $\pi$ in a dictionary and include their relative difference.

In [ ]:
t.reset()
t.speed('fastest')
perim,n = 600,48
pi_dict = {}      # define an empty dictionary
for iter in range(5):
    n *= 2       # double n every iteration
    t.home()
    for k in range(n//2):
        t.forward(perim / n)
        t.left(360 / n)
    pi_high = perim / t.ycor()
    pi_low = perim / t.distance(0,0)
    pi_dict[n] = [pi_low, pi_high, pi_high/pi_low - 1]
    print('Computation with',n, 'sides shows that', round(pi_dict[n][0],7), '< π <', round(pi_dict[n][1],7))
precision1536 = 1 / pi_dict[1536][2]  # precision is reciprocal of relative difference 
print('Computation with 1536 sides specifies π to one part in', round(precision1536/1E6,1), 'million')

In [ ]:
# repeatedly quadruple steps to tighten bounds on π 
import math
radius,steps = 200,40
for iter in range(6):
    steps *= 4
    dx = radius/steps
    x = [-radius + i * dx  for i in range(steps+1)]
    y = [math.sqrt(radius * radius - val * val) for val in x]
    area_high = sum(y[1:]) * dx  
    area_low = sum(y[:-1]) * dx  
    pi_high = area_high * 4 / (radius * radius)
    pi_low = area_low * 4 / (radius * radius)
    rel_diff = (pi_high - pi_low) / math.pi
    # any terms with parentheses around them can be extended to multiple lines
    print(steps,'steps:',round(pi_low,7),'< π <',round(pi_high,7),   # continues on next line
                '\nrelative difference',round(rel_diff,6),'\n')